In [ ]:
# ============================================================
# DETERMINISM SETUP — must execute BEFORE any CUDA op / model load.
# Reason: cublas workspace config and use_deterministic_algorithms()
# must be set on a *fresh* CUDA context. If you later see a non-
# deterministic kernel error, switch warn_only=False -> True.
# ============================================================
import os
os.environ["PYTHONHASHSEED"]         = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # required by torch.use_deterministic_algorithms

import torch
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
torch.use_deterministic_algorithms(True, warn_only=True)

torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print("Determinism flags set. cuDNN deterministic =",
      torch.backends.cudnn.deterministic,
      "| benchmark =", torch.backends.cudnn.benchmark)


In [ ]:
# ============================================================
# CONFIG + HELPERS
# Aligned with the 0-shot and few-shot v4 notebooks.
# ============================================================
import os, re, gc, json, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix,
)

# ---------- Paths ----------
BASE_PATH   = "."  # repo root
EXCEL_PATH  = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = "DATA/outputs/predictions/baselines"  # OOF .npy shipped here
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ---------- Reproducibility ----------
SEED        = 42
N_SPLITS    = 5

# ---------- Model / training ----------
MODEL_NAME   = "BAAI/bge-reranker-v2-m3"
MAX_LEN      = 512
BATCH_TRAIN  = 8
BATCH_EVAL   = 16
EPOCHS       = 3
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

# IMPORTANT: bf16 GEMM kernels on A100 are NOT bit-deterministic
# even with cudnn.deterministic=True. We disable bf16 to make the
# 5-fold OOF reproducible across runs. Cost: ~2-3x slower per fold,
# still well under 2 minutes per fold on an A100.
USE_BF16     = False


# ============================================================
# Reproducibility helper
# Called ONCE up-front and again at the start of every fold so
# that fold k always starts from the same RNG state regardless
# of what happened in folds 0..k-1.
# ============================================================
def set_all_seeds(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # cudnn flags already set in the determinism cell, but re-assert here
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_all_seeds(SEED)


# ============================================================
# GPU helper
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# ============================================================
# Labels + Gold (verbatim from 0-shot / few-shot v4)
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]:  return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading dataset from {path}...")
    df = pd.read_csv(path)
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")
    if "pred_art" not in df.columns:
        df["pred_art"] = ""

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)
    df = df.dropna(subset=["text", "article_text", "lbl_Gold"]).copy().reset_index(drop=True)
    df["pred_art"] = df["pred_art"].fillna("").astype(str)

    print(f"    Total: {len(df)}")
    print(f"    Gold yes : {int((df['lbl_Gold']==1).sum())}")
    print(f"    Gold no  : {int((df['lbl_Gold']==0).sum())}")
    return df


# ============================================================
# Folding (verbatim from few-shot v4)
# ============================================================
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out


# ============================================================
# Cross-encoder pair dataset
# ============================================================
class PairDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.articles  = df["article_text"].astype(str).tolist()
        self.chunks    = df["text"].astype(str).tolist()
        self.labels    = df["lbl_Gold"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.articles[idx],
            self.chunks[idx],
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_tensors=None,
        )
        enc["labels"] = self.labels[idx]
        return enc


def collate_fn(batch, tokenizer):
    max_l = max(len(b["input_ids"]) for b in batch)
    pad_id = tokenizer.pad_token_id
    input_ids, attn, labels = [], [], []
    for b in batch:
        pad_n = max_l - len(b["input_ids"])
        input_ids.append(b["input_ids"]      + [pad_id] * pad_n)
        attn.append(    b["attention_mask"] + [0]      * pad_n)
        labels.append(  b["labels"])
    return {
        "input_ids":      torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attn,      dtype=torch.long),
        "labels":         torch.tensor(labels,    dtype=torch.long),
    }


# ============================================================
# Train one fold (fresh model)
# ============================================================
def _seed_worker(worker_id):
    # Required for deterministic DataLoader workers if num_workers > 0.
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def train_one_fold(df_train, df_test, tokenizer, device, fold_id):
    print(f"\n--- Fold {fold_id}: train={len(df_train)}, test={len(df_test)} ---")

    # Re-seed at the start of every fold so fold k is independent of
    # how much randomness was consumed in folds 0..k-1.
    fold_seed = SEED + fold_id
    set_all_seeds(fold_seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        ignore_mismatched_sizes=True,   # BGE-reranker has a 1-output head; we replace it with a 2-class head
        torch_dtype=(torch.bfloat16 if (USE_BF16 and torch.cuda.is_available()) else torch.float32),
    ).to(device)
    model.train()

    train_ds = PairDataset(df_train, tokenizer)
    test_ds  = PairDataset(df_test,  tokenizer)

    # Explicit generator for the train shuffler — pinned to fold_seed.
    g = torch.Generator()
    g.manual_seed(fold_seed)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_TRAIN, shuffle=True,
        collate_fn=lambda b: collate_fn(b, tokenizer),
        num_workers=0, pin_memory=True,
        generator=g,
        worker_init_fn=_seed_worker,
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_EVAL, shuffle=False,
        collate_fn=lambda b: collate_fn(b, tokenizer),
        num_workers=0, pin_memory=True,
    )

    no_decay = ["bias", "LayerNorm.weight"]
    head_keywords = ["classifier"]

    def is_head(n):
        return any(h in n for h in head_keywords)

    grouped = [
        {"params": [p for n, p in model.named_parameters()
                    if not is_head(n) and not any(nd in n for nd in no_decay)],
         "lr": 2e-5, "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters()
                    if not is_head(n) and any(nd in n for nd in no_decay)],
         "lr": 2e-5, "weight_decay": 0.0},
        {"params": [p for n, p in model.named_parameters() if is_head(n)],
         "lr": 1e-4, "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(grouped)
    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        pbar = tqdm(train_loader, desc=f"  epoch {epoch+1}/{EPOCHS}")
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            running += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.3f}")
        print(f"    epoch {epoch+1} mean loss: {running/len(train_loader):.4f}")

    # ----- Evaluate on test fold -----
    model.eval()
    all_probs_yes, all_preds = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="  eval"):
            labels = batch.pop("labels")
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits.float()
            probs  = torch.softmax(logits, dim=-1)[:, 1]
            preds  = logits.argmax(dim=-1)
            all_probs_yes.extend(probs.cpu().numpy().tolist())
            all_preds.extend(  preds.cpu().numpy().tolist())

    del model, optimizer, scheduler
    _clear_cuda()

    return np.array(all_probs_yes, dtype=float), np.array(all_preds, dtype=int)


# ============================================================
# Metrics + confusion (verbatim shape from 0-shot)
# ============================================================
def calculate_metrics(y_true, y_pred, name="GOLD"):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    acc      = accuracy_score(y_true, y_pred)
    mcc      = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_pred)) > 1 else 0.0
    prec_oui = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec_oui  = recall_score(   y_true, y_pred, pos_label=1, zero_division=0)
    f1_oui   = f1_score(       y_true, y_pred, pos_label=1, zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fpr      = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    yes_rate = float(np.mean(y_pred == 1))

    return {
        "Target":          name,
        "N_Valid":         int(len(y_true)),
        "Yes_pct":         round(100.0 * yes_rate, 2),
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui":   round(prec_oui, 4),
        "Recall_Oui":      round(rec_oui, 4),
        "F1_Oui":          round(f1_oui, 4),
        "FPR_pct":         round(100.0 * fpr, 2),
        "MCC":             round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp + fn) > 0 else "0/0",
    }, cm

def print_confusion(cm):
    tn, fp = cm[0]; fn, tp = cm[1]
    print("Confusion matrix (rows=true, columns=predicted) [0=no, 1=yes]")
    print(f"          pred_no   pred_yes")
    print(f"true_no     {tn:6d}   {fp:6d}")
    print(f"true_yes    {fn:6d}   {tp:6d}")


In [ ]:
# ============================================================
# MAIN — load data once, fold once, train + eval per fold,
# aggregate OOF predictions, save metrics + OOF dataframe.
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | Model: {MODEL_NAME} | bf16={USE_BF16}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

df_raw = load_and_prep_data(EXCEL_PATH)
df_cv  = make_grouped_folds(df_raw, n_splits=N_SPLITS, seed=SEED)
print(f"[Folds] sizes: {df_cv.groupby('fold').size().tolist()}")

print(f"\n[2] Loading tokenizer ({MODEL_NAME})...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

n = len(df_cv)
oof_probs = np.full(n, np.nan, dtype=float)
oof_preds = np.full(n, -1,     dtype=int)

for fid in range(N_SPLITS):
    train_mask = (df_cv["fold"] != fid).values
    test_mask  = (df_cv["fold"] == fid).values

    df_train = df_cv[train_mask].copy()
    df_test  = df_cv[test_mask].copy()

    probs_yes, preds = train_one_fold(df_train, df_test, tokenizer, device, fid)

    test_idx = df_cv.index[test_mask].values
    oof_probs[test_idx] = probs_yes
    oof_preds[test_idx] = preds

# ============================================================
# Aggregate metrics (Gold only)
# ============================================================
df_cv["bge_ft_prob_yes"] = oof_probs
df_cv["bge_ft_pred"]     = oof_preds

print("\n" + "=" * 70)
print("RESULTS — BGE-reranker-v2-m3 fine-tuned (5-fold OOF)")
print("=" * 70)
metrics, cm = calculate_metrics(
    df_cv["lbl_Gold"].values.astype(int),
    oof_preds,
    name="GOLD",
)
for k, v in metrics.items():
    print(f"  {k:18s}: {v}")
print()
print_confusion(cm)


In [ ]:
# ============================================================
# Save OOF + metrics
# ============================================================
ts = datetime.now().strftime("%Y%m%d_%H%M")

cols = [
    "decision_id", "chunk_id", "pred_art", "fold",
    "text", "article_text",
    "eval_A2", "eval_A1", "eval_A3",
    "lbl_A2", "lbl_A1", "lbl_Gold",
    "bge_ft_prob_yes", "bge_ft_pred",
]
oof_df = df_cv[[c for c in cols if c in df_cv.columns]].copy()

def status(r):
    g, p = r["lbl_Gold"], r["bge_ft_pred"]
    if pd.isna(g) or p == -1: return "Ignore"
    if g == 1 and p == 1: return "TP"
    if g == 0 and p == 1: return "FP"
    if g == 1 and p == 0: return "FN"
    if g == 0 and p == 0: return "TN"
    return "Error"
oof_df["Error_Type_vs_Gold"] = oof_df.apply(status, axis=1)

oof_path     = os.path.join(OUTPUT_PATH, f"OOF_bge_ft_Gold_{ts}.xlsx")
oof_df.to_excel(oof_path, index=False)

metrics_df   = pd.DataFrame([{**metrics, "Model": "BGE-reranker-v2-m3-FT"}])
metrics_path = os.path.join(OUTPUT_PATH, f"metrics_bge_ft_{ts}.xlsx")
metrics_df.to_excel(metrics_path, index=False)

print("Saved:")
print(f"  OOF     : {oof_path}")
print(f"  Metrics : {metrics_path}")


In [ ]:
# ============================================================
# FP analysis — does the fine-tuned BGE concentrate its errors
# on annotator-disagreement cases, like the supervised models
# in Table 11 of the paper?
# Methodology aligned: Wald on log(OR) via Table2x2.
# ============================================================
import glob
from statsmodels.stats.contingency_tables import Table2x2

oof_files = sorted(glob.glob(os.path.join(OUTPUT_PATH, "OOF_bge_ft_Gold_*.xlsx")))
assert len(oof_files) > 0, "No OOF file found — run the training cells first."
oof_path = oof_files[-1]
print(f"[Reading] {oof_path}")
df_oof = pd.read_excel(oof_path)

agree = (
    df_oof["lbl_A2"].notna()
    & df_oof["lbl_A1"].notna()
    & (df_oof["lbl_A2"] == df_oof["lbl_A1"])
).astype(int).values
gold  = df_oof["lbl_Gold"].values
preds = df_oof["bge_ft_pred"].values.astype(int)

print(f"\nDataset      : n = {len(df_oof)}")
print(f"  Agreement  : {int(agree.sum())} ({100*agree.mean():.1f}%)")
print(f"  Disagreement: {int((1-agree).sum())} ({100*(1-agree).mean():.1f}%)")

neg = (gold == 0)
ag  = neg & (agree == 1)
dis = neg & (agree == 0)

fp_ag  = int(((preds == 1) & ag).sum())
fp_dis = int(((preds == 1) & dis).sum())
n_ag   = int(ag.sum())
n_dis  = int(dis.sum())

fpr_ag  = fp_ag  / n_ag  if n_ag  else float("nan")
fpr_dis = fp_dis / n_dis if n_dis else float("nan")

table = [[fp_dis, n_dis - fp_dis],
         [fp_ag,  n_ag  - fp_ag]]
t22 = Table2x2(table)
OR = t22.oddsratio
p_wald = t22.oddsratio_pvalue()
ci_lo, ci_hi = np.exp(t22.log_oddsratio_confint(alpha=0.05))

print("\n" + "=" * 70)
print("FALSE POSITIVES BY ANNOTATOR AGREEMENT (BGE-reranker-v2-m3 fine-tuned)")
print("=" * 70)
print(f"{'Subset':<14}{'Cases':>8}{'TN+FP':>8}{'FP':>6}{'FPR':>8}")
print("-" * 70)
print(f"{'Agree':<14}{int(agree.sum()):>8}{n_ag:>8}{fp_ag:>6}{100*fpr_ag:>7.1f}%")
print(f"{'Disagree':<14}{int((1-agree).sum()):>8}{n_dis:>8}{fp_dis:>6}{100*fpr_dis:>7.1f}%")
print(f"{'Total':<14}{len(df_oof):>8}{n_ag+n_dis:>8}{fp_ag+fp_dis:>6}"
      f"{100*(fp_ag+fp_dis)/(n_ag+n_dis):>7.1f}%")
print(f"\nOdds ratio (FPR_dis / FPR_ag) = {OR:.2f}  (95% CI [{ci_lo:.2f}, {ci_hi:.2f}])")
print(f"p-value (Wald on log(OR), two-sided) = {p_wald:.4f}")

total_fp = fp_ag + fp_dis
share_dis = (100 * fp_dis / total_fp) if total_fp else 0.0
if total_fp:
    print(f"\nOf {total_fp} total FPs, {fp_dis} ({share_dis:.0f}%) fall on the "
          f"{100*(1-agree.mean()):.0f}% of cases where annotators disagreed.")

import datetime as _dt
ts = _dt.datetime.now().strftime("%Y%m%d_%H%M")
fpr_row = pd.DataFrame([{
    "Model":          "BGE-reranker-v2-m3-FT",
    "FP_agree":       fp_ag,
    "N_neg_agree":    n_ag,
    "FPR_agree":      round(fpr_ag,  4) if fpr_ag  == fpr_ag  else None,
    "FP_disagree":    fp_dis,
    "N_neg_disagree": n_dis,
    "FPR_disagree":   round(fpr_dis, 4) if fpr_dis == fpr_dis else None,
    "OR_dis_vs_ag":   round(OR,      3) if OR      == OR      else None,
    "CI95_low":       round(ci_lo,   3) if ci_lo   == ci_lo   else None,
    "CI95_high":      round(ci_hi,   3) if ci_hi   == ci_hi   else None,
    "p_wald":         round(p_wald,  4) if p_wald  == p_wald  else None,
    "Total_FP":       total_fp,
    "Share_FP_on_disagree_pct": round(share_dis, 1) if total_fp else None,
}])
fpr_path = os.path.join(OUTPUT_PATH, f"fpr_by_agreement_bge_ft_{ts}.xlsx")
fpr_row.to_excel(fpr_path, index=False)
print(f"\nSaved: {fpr_path}")


In [ ]:
# ============================================================
# Save OOF probabilities to .npy
# Format identical to ST-MiniLM.npy: np.float64, shape (n,),
# values = probabilities of the positive class (YES).
# Order = df_cv row order (so consistent with the other .npy files
# if they follow the same sort convention).
# ============================================================
import glob

NPY_NAME = 'bge-reranker-ft.npy'   # rename if needed to match the naming convention

# Retrieve probabilities: from df_cv if still in memory, otherwise from the latest OOF xlsx
try:
    p_oof = df_cv['bge_ft_prob_yes'].values.astype(np.float64)
    src = 'df_cv (memory)'
except NameError:
    oof_files = sorted(glob.glob(os.path.join(OUTPUT_PATH, "OOF_bge_ft_Gold_*.xlsx")))
    assert len(oof_files) > 0, "No OOF file found — run the training cells first."
    p_oof = pd.read_excel(oof_files[-1])['bge_ft_prob_yes'].values.astype(np.float64)
    src = oof_files[-1]

npy_path = os.path.join(OUTPUT_PATH, NPY_NAME)
np.save(npy_path, p_oof)

print(f"Source  : {src}")
print(f"Saved   : {npy_path}")
print(f"  shape : {p_oof.shape}, dtype : {p_oof.dtype}")
print(f"  min={p_oof.min():.3f}, max={p_oof.max():.3f}, mean={p_oof.mean():.3f}")
print(f"  NaNs  : {int(np.isnan(p_oof).sum())}")